# FINS notebook 4 — the weak form through the learned heat operators (Colab, GPU)

Registered in `README_EXPERIMENTS.md` (EX4, 07.10) before any of its runs: FINS weak (exp51's frozen recipe) through
the protocol's heat CNOs `S_theta_heat_s{42,43,44}_g64x128_cno_smooth_L1.pt`, clean data, scored later on the finite
algebra, the non-trivial generators and heat's class. Three runs; on one CPU core each takes about 2.5 h, here
minutes.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down). The
CNOs are already in `MyDrive/fins_runs/results` from the protocol runs; cell 1 looks for them elsewhere on the Drive
if not. Cell 3 zips the three outputs into `MyDrive/fins_nb4.zip` (a few MB) for the chat.

In [ ]:
# 1. setup: Drive, the newest repository zip with exp55, the protocol's heat CNOs, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp55(z):
    with zipfile.ZipFile(z) as zz:
        return 'fins_repo/experiments/exp55_extra.py' in zz.namelist()


ZIP_PATH = newest('fins_repo_full*.zip', has_exp55)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp55_extra.py: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
for s in (42, 43, 44):
    name = f'S_theta_heat_s{s}_g64x128_cno_smooth_L1.pt'
    dst = os.path.join(R, name)
    if os.path.exists(dst):
        print('found', name)
        continue
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    if not hits:
        raise SystemExit(f'{name} is not on the Drive: it is the protocol heat CNO of seed {s}')
    shutil.copy(hits[0], dst)
    print('copied', hits[0], '->', dst)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'torchdiffeq'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')

In [ ]:
# 2. the three registered runs (finished ones are skipped after a disconnect: rerun cell 1, then this cell)
t0 = time.time()
cmd = [sys.executable, '-u', 'exp55_extra.py', 'run', '--parts', 'heat-learned', '--seeds', '42,43,44', '--runs', R,
       '--threads', '2']
with subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end='', flush=True)
print(f'exit {p.returncode} after {time.time() - t0:.0f} s')
for s in (42, 43, 44):
    log = os.path.join(R, 'noise_weak_runs', 'clean', 'logs', f'weak-learned_heat_s{s}.log')
    if os.path.exists(log):
        keep = [l.rstrip() for l in open(log) if 'algebra inside' in l or 'own count' in l or 'weak blocks' in l]
        print(f'seed {s}:', *[k[:150] for k in keep[-3:]], sep='\n  ')

In [ ]:
# 3. zip the outputs for the chat: MyDrive/fins_nb4.zip
out = '/content/drive/MyDrive/fins_nb4.zip'
d = os.path.join(R, 'noise_weak_runs', 'clean')
files = sorted(glob.glob(os.path.join(d, 'weak-learned_heat_s4*.npz')) +
               glob.glob(os.path.join(d, 'logs', 'weak-learned_heat_s4*.log')))
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.relpath(f, R))
print(out, os.path.getsize(out), 'bytes;', len(files), 'files:', *[os.path.relpath(f, R) for f in files], sep='\n  ')